In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

In [2]:
spark = SparkSession.builder.appName('OlistData').getOrCreate()

26/02/14 14:06:14 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [3]:
hdfs_path = '/data/olist/'

In [4]:
customers_df = spark.read.csv(hdfs_path + 'olist_customers_dataset.csv', header=True, inferSchema = True)
geo_df = spark.read.csv(hdfs_path + 'olist_geolocation_dataset.csv', header=True, inferSchema = True)
order_items_df = spark.read.csv(hdfs_path + 'olist_order_items_dataset.csv', header=True, inferSchema = True)
order_payments_df = spark.read.csv(hdfs_path + 'olist_order_payments_dataset.csv', header=True, inferSchema = True)
order_reviews_df = spark.read.csv(hdfs_path + 'olist_order_reviews_dataset.csv', header=True, inferSchema = True)
orders_df = spark.read.csv(hdfs_path + 'olist_orders_dataset.csv', header=True, inferSchema = True)
products_df = spark.read.csv(hdfs_path + 'olist_products_dataset.csv', header=True, inferSchema = True)
sellers_df = spark.read.csv(hdfs_path + 'olist_sellers_dataset.csv', header=True, inferSchema = True)
product_category_df = spark.read.csv(hdfs_path + 'product_category_name_translation.csv', header=True, inferSchema = True)

## Identify Missing Values

In [5]:
def missing_values(df, df_name):
    print(f"Missing values in {df_name}")
    df.select([count(when(col(c).isNull(), 1)).alias(c) for c in df.columns]).show()

In [6]:
missing_values(orders_df, "Order Table")

Missing values in Order Table
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|order_id|customer_id|order_status|order_purchase_timestamp|order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|       0|          0|           0|                       0|              160|                        1783|                         2965|                            0|
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+



In [7]:
missing_values(customers_df, "Customer Table")

Missing values in Customer Table
+-----------+------------------+------------------------+-------------+--------------+
|customer_id|customer_unique_id|customer_zip_code_prefix|customer_city|customer_state|
+-----------+------------------+------------------------+-------------+--------------+
|          0|                 0|                       0|            0|             0|
+-----------+------------------+------------------------+-------------+--------------+



In [8]:
missing_values(order_items_df, "Order Item Table")

Missing values in Order Item Table
+--------+-------------+----------+---------+-------------------+-----+-------------+
|order_id|order_item_id|product_id|seller_id|shipping_limit_date|price|freight_value|
+--------+-------------+----------+---------+-------------------+-----+-------------+
|       0|            0|         0|        0|                  0|    0|            0|
+--------+-------------+----------+---------+-------------------+-----+-------------+



## Handle Missing Values

### 1. Drop Missing Values --> for non critical column
### 2. Fill the missing values --> for numerical columns
### 3. Impute missing values --> for continuous data

## Drop Missing Values

In [9]:
orders_df_cleaned = orders_df.na.drop(subset = ['order_id', 'customer_id', 'order_status'])

## Filling Missing values

In [10]:
orders_df_cleaned = orders_df.fillna({'order_delivered_customer_date':'9999-12-31'})

## Impute Missing values

In [11]:
from pyspark.ml.feature import Imputer

In [12]:
imputer = Imputer(inputCols=['payment_value'], outputCols=['payment_value_imputed'], strategy='mean')

In [13]:
payments_df_cleaned = imputer.fit(order_payments_df).transform(order_payments_df)

In [14]:
payments_df_cleaned.show()

+--------------------+------------------+------------+--------------------+-------------+---------------------+
|            order_id|payment_sequential|payment_type|payment_installments|payment_value|payment_value_imputed|
+--------------------+------------------+------------+--------------------+-------------+---------------------+
|b81ef226f3fe1789b...|                 1| credit_card|                   8|        99.33|                99.33|
|a9810da82917af2d9...|                 1| credit_card|                   1|        24.39|                24.39|
|25e8ea4e93396b6fa...|                 1| credit_card|                   1|        65.71|                65.71|
|ba78997921bbcdc13...|                 1| credit_card|                   8|       107.78|               107.78|
|42fdf880ba16b47b5...|                 1| credit_card|                   2|       128.45|               128.45|
|298fcdf1f73eb413e...|                 1| credit_card|                   2|        96.12|               

## Standardizing the Format

In [15]:
def print_schema(df, df_name):
    print(f"Schema of {df_name}")
    df.printSchema()

In [16]:
print_schema(customers_df, "Customers")

Schema of Customers
root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: integer (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)



In [17]:
orders_df_cleaned = orders_df_cleaned.withColumn('order_purchase_timestamp', to_date(col('order_purchase_timestamp')))

In [18]:
payments_df_cleaned = payments_df_cleaned.withColumn('payment_type', when(col('payment_type')=='boleto', 'Bank Transfer')
                                                     .when(col('payment_type')=='credit_card', 'Credit Card')
                                                     .when(col('payment_type')=='debit_card', 'Debit Card')
                                                     .otherwise('other'))

In [19]:
customers_df_cleaned = customers_df.withColumn('customer_zip_code_prefix', col('customer_zip_code_prefix').cast('string'))

In [20]:
customers_df_cleaned.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: string (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)



## Remove Duplicate Records

In [21]:
customers_df_cleaned = customers_df_cleaned.dropDuplicates(['customer_id'])

## Joining Datasets

In [22]:
order_with_details = orders_df_cleaned.join(order_items_df, 'order_id', 'left')\
                    .join(order_payments_df, 'order_id', 'left')\
                    .join(customers_df_cleaned, 'customer_id', 'left')

## Feature Engineering

In [23]:
order_with_total_value = order_with_details.groupBy('order_id').agg(sum('payment_value').alias('Total Order Value'))

In [24]:
order_with_total_value.show(5)

+--------------------+-----------------+
|            order_id|Total Order Value|
+--------------------+-----------------+
|118045506e1c1dda0...|           1802.0|
|f44cb69655f8e4d13...|           164.32|
|edcc6b79e8394346b...|           162.63|
|9f98d6530155e3b38...|           316.76|
|949280c70c6d62ec9...|            49.42|
+--------------------+-----------------+
only showing top 5 rows



In [25]:
## Delivery Time Calculation

order_with_delivery_time = order_with_details\
                        .withColumn('delivery_time', datediff(col('order_delivered_customer_date'), col('order_purchase_timestamp')))

## Advance Transformation

In [26]:
order_items_df.show(5)

+--------------------+-------------+--------------------+--------------------+-------------------+-----+-------------+
|            order_id|order_item_id|          product_id|           seller_id|shipping_limit_date|price|freight_value|
+--------------------+-------------+--------------------+--------------------+-------------------+-----+-------------+
|00010242fe8c5a6d1...|            1|4244733e06e7ecb49...|48436dade18ac8b2b...|2017-09-19 09:45:35| 58.9|        13.29|
|00018f77f2f0320c5...|            1|e5f2d52b802189ee6...|dd7ddc04e1b6c2c61...|2017-05-03 11:05:13|239.9|        19.93|
|000229ec398224ef6...|            1|c777355d18b72b67a...|5b51032eddd242adc...|2018-01-18 14:48:30|199.0|        17.87|
|00024acbcdf0a6daa...|            1|7634da152a4610f15...|9d7a1d34a50524090...|2018-08-15 10:10:18|12.99|        12.79|
|00042b26cf59d7ce6...|            1|ac6c3623068f30de0...|df560393f3a51e745...|2017-02-13 13:57:51|199.9|        18.14|
+--------------------+-------------+------------

In [30]:
quantiles = order_items_df.approxQuantile('price', [0.01, 0.99], 0.0)
low_cutoff, high_cutoff = quantiles[0], quantiles[1]

In [32]:
order_items_df.select('price').summary().show()

+-------+------------------+
|summary|             price|
+-------+------------------+
|  count|            112650|
|   mean|120.65373901471354|
| stddev|183.63392805026012|
|    min|              0.85|
|    25%|              39.9|
|    50%|             74.99|
|    75%|             134.9|
|    max|            6735.0|
+-------+------------------+



In [31]:
low_cutoff, high_cutoff

(9.99, 890.0)

In [36]:
order_items_df_cleaned = order_items_df.filter((col('price') >= low_cutoff) & (col('price') <= high_cutoff))

In [39]:
payments_df_cleaned.select('payment_installments').summary().show()

+-------+--------------------+
|summary|payment_installments|
+-------+--------------------+
|  count|              103886|
|   mean|   2.853348863176944|
| stddev|  2.6870506738564925|
|    min|                   0|
|    25%|                   1|
|    50%|                   1|
|    75%|                   4|
|    max|                  24|
+-------+--------------------+



In [43]:
products_df_cleaned = products_df\
                    .withColumn('product_size_category', when(col('product_weight_g') < 500, 'Small')\
                                .when(col('product_weight_g').between(500, 2000), 'Medium').otherwise('Large'))

In [44]:
products_df_cleaned.show(5)

+--------------------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+---------------------+
|          product_id|product_category_name|product_name_lenght|product_description_lenght|product_photos_qty|product_weight_g|product_length_cm|product_height_cm|product_width_cm|product_size_category|
+--------------------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+---------------------+
|1e9e8ef04dbcff454...|           perfumaria|                 40|                       287|                 1|             225|               16|               10|              14|                Small|
|3aa071139cb16b67c...|                artes|                 44|                       276|                 1|            1000|               30|               18|              20|        

In [47]:
## Calculate Total Revenue Per Seller

In [57]:
orders_sellers_df = sellers_df.join(order_items_df_cleaned, 'seller_id', 'left')\
                    .join(orders_df_cleaned, 'order_id', 'left').join(order_payments_df, 'order_id', 'left')

In [62]:
total_revenue_per_seller = orders_sellers_df.groupBy('seller_id').agg(round(sum('payment_value'), 2).alias('total_revenue_per_seller'))
total_revenue_per_seller.orderBy('total_revenue_per_seller', ascending = False).show()

+--------------------+------------------------+
|           seller_id|total_revenue_per_seller|
+--------------------+------------------------+
|7c67e1448b00f6e96...|               507166.91|
|1025f0e2d44d7041d...|               308222.04|
|4a3ca9315b744ce9f...|               301245.27|
|1f50f920176fa81da...|               290253.42|
|da8622b14eb17ae28...|               271447.96|
|4869f7a5dfa277a7d...|               263234.55|
|955fee9216a65b617...|                236322.3|
|6560211a19b47992c...|               179657.75|
|fa1c13f2614d7b5c4...|               174717.53|
|7a67c85e85bb2ce85...|                169030.8|
|25c5c91f63607446a...|               160534.74|
|a1043bafd471dff53...|               154356.91|
|53243585a1d6dc264...|               148210.15|
|620c87c171fb2a6dd...|               145267.95|
|46dc3b2cc0980fb8e...|                140931.2|
|cc419e0650a3c5ba7...|               140200.18|
|3d871de0142ce09b7...|               131982.15|
|7d13fca1522535862...|               129

In [68]:
!hdfs dfs -ls /data/olist_proc

Found 2 items
drwxr-xr-x   - root hadoop          0 2026-02-14 14:38 /data/olist_proc/cleaned_data.parquet
drwxr-xr-x   - root hadoop          0 2026-02-14 14:43 /data/olist_proc/product_data_cleaned.parquet


In [66]:
order_with_details.write.mode('overwrite').parquet('/data/olist_proc/cleaned_data.parquet')

In [67]:
products_df_cleaned.write.mode('overwrite').parquet('/data/olist_proc/product_data_cleaned.parquet')